# Kaggle Model + CSV Benchmark and FastAPI Lab
Attach model artifacts and a CSV using **Add Input**, then run the cell below.


In [ ]:
# KAGGLE MODEL + CSV BENCHMARK + FASTAPI — SINGLE CELL
# Attach actual trained model files and a matching evaluation CSV via Add Input.
# Accepted model files ONLY: .joblib and .pt. Dataset files ONLY: .csv.
# .pt must be TorchScript for standalone inference; weight-only checkpoints
# need their original nn.Module class and cannot be served automatically.
# Load only .joblib artifacts you trust.
import json, time, io
from pathlib import Path
import numpy as np
import pandas as pd
import joblib, torch
import ipywidgets as w
from IPython.display import display, clear_output
from sklearn.metrics import accuracy_score, f1_score, mean_squared_error, r2_score
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel

ROOT = Path('/kaggle/input') if Path('/kaggle/input').exists() else Path('.')
OUT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')
OUT.mkdir(parents=True, exist_ok=True)
MODELS = {}

# Strict filters: never mistake a manifest or preprocessor for a model.
def find_models():
    banned = ('preprocessor', 'preprocess', 'encoder', 'scaler', 'imputer', 'tokenizer')
    return sorted((p for p in ROOT.rglob('*') if p.is_file()
        and p.suffix.lower() in {'.pt', '.joblib'}
        and not any(word in p.stem.lower() for word in banned)), key=str)

def find_csvs():
    # Ignore CSVs in model folders; show dataset CSVs and standalone CSV inputs.
    return sorted((p for p in ROOT.rglob('*.csv') if p.is_file()
        and '/models/' not in p.as_posix().lower()), key=str)

models_ui = w.SelectMultiple(description='Models', rows=7, layout=w.Layout(width='99%'))
csv_ui = w.Dropdown(description='Dataset', layout=w.Layout(width='99%'))
target_ui = w.Dropdown(description='Target')
task_ui = w.Dropdown(description='Task', options=['classification', 'regression'])
refresh_btn = w.Button(description='Refresh files')
load_btn = w.Button(description='Load models', button_style='success')
check_btn = w.Button(description='Check inputs')
bench_btn = w.Button(description='Benchmark', button_style='primary')
query_btn = w.Button(description='Query FastAPI')
log = w.Output()

def set_targets(_=None):
    previous = target_ui.value
    try:
        cols = pd.read_csv(csv_ui.value, nrows=2).columns.tolist()
        target_ui.options = cols
        target_ui.value = previous if previous in cols else (cols[-1] if cols else None)
    except Exception:
        target_ui.options = []

def refresh(_=None):
    m, d = find_models(), find_csvs()
    models_ui.options = [(f'{p.name} | {p.parent}', str(p)) for p in m]
    csv_ui.options = [(f'{p.name} | {p.parent}', str(p)) for p in d]
    set_targets()
    with log:
        clear_output(wait=True)
        print(f'Found {len(m)} model files (.pt/.joblib), {len(d)} dataset CSV files.')
        if not m: print('Attach actual model artifacts using Kaggle Add Input.')
        if not d: print('Attach your evaluation CSV under Kaggle Add Input > Datasets.')

def load_one(path):
    p = Path(path)
    if p.suffix.lower() == '.joblib':
        obj = joblib.load(p)  # Trusted artifacts ONLY.
        if not callable(getattr(obj, 'predict', None)):
            raise ValueError(f'{type(obj).__name__} has no predict(); not a trained estimator.')
        return obj, 'estimator'
    try:
        obj = torch.jit.load(str(p), map_location='cpu')
        obj.eval()
        return obj, 'torchscript'
    except Exception as torchscript_error:
        # Inspect weights safely; DO NOT execute arbitrary pickle code.
        try:
            ckpt = torch.load(p, map_location='cpu', weights_only=True)
            if isinstance(ckpt, dict):
                keys = list(ckpt.keys())[:8]
                raise ValueError('PyTorch weight checkpoint, not TorchScript; '
                    f'requires original nn.Module architecture. Keys: {keys}')
            raise ValueError(f'Unsupported .pt object: {type(ckpt).__name__}')
        except ValueError:
            raise
        except Exception:
            raise ValueError('Not a loadable TorchScript file. If it is a checkpoint, '
                'export as TorchScript from the original training notebook.') from torchscript_error

def load(_=None):
    MODELS.clear()
    with log:
        clear_output(wait=True)
        for path in models_ui.value:
            try:
                obj, kind = load_one(path)
                name = Path(path).stem
                n = 2
                while name in MODELS:
                    name = f'{Path(path).stem}_{n}'; n += 1
                MODELS[name] = {'obj': obj, 'kind': kind, 'path': path}
                print('LOADED:', name, '|', kind, '|', path)
            except Exception as e:
                print('NOT LOADED:', path, '\n ', e)
        print('Models available for inference:', len(MODELS))

def data():
    if not csv_ui.value: raise ValueError('Choose one CSV dataset.')
    df = pd.read_csv(csv_ui.value)
    target = target_ui.value
    if target not in df.columns: raise ValueError('Choose the CSV target column.')
    df = df.dropna(subset=[target])
    if df.empty: raise ValueError('CSV has no labeled records.')
    X, y = df.drop(columns=[target]), df[target]
    if X.empty: raise ValueError('CSV needs at least one feature column.')
    return X, y

def predict(name, X):
    entry = MODELS[name]
    obj, kind = entry['obj'], entry['kind']
    if kind == 'estimator':
        if hasattr(obj, 'feature_names_in_'):
            needed = list(obj.feature_names_in_)
            missing = [c for c in needed if c not in X.columns]
            if missing: raise ValueError(f'Missing columns {missing}; CSV has {list(X.columns)}')
            X = X.loc[:, needed]
        return np.asarray(obj.predict(X)).reshape(-1)
    # TorchScript typically needs numeric features in the exact training order.
    try:
        numbers = X.to_numpy(dtype=np.float32)
    except (ValueError, TypeError):
        raise ValueError('TorchScript needs numeric, preprocessed features. '
            'This CSV contains text/categorical data; use the training preprocessor first.')
    with torch.no_grad():
        result = obj(torch.from_numpy(numbers))
        if isinstance(result, (tuple, list)): result = result[0]
        raw = result.detach().cpu().numpy()
    if task_ui.value == 'regression': return raw.reshape(-1)
    if raw.ndim == 2 and raw.shape[1] > 1: return raw.argmax(axis=1)
    # Assumes binary probabilities, not logits.
    return (raw.reshape(-1) >= 0.5).astype(int)

def check(_=None):
    with log:
        clear_output(wait=True)
        try:
            X, y = data()
            print('CSV:', csv_ui.value, '| rows:', len(X), '| target:', target_ui.value)
            print('Feature columns:', list(X.columns))
            if not MODELS: print('Select models and click Load models first.')
            for name in MODELS:
                try: print('PASS:', name, '=>', predict(name, X.iloc[:1]).tolist())
                except Exception as e: print('FAIL:', name, '=>', e)
        except Exception as e: print('INPUT ERROR:', e)

def benchmark():
    if not MODELS: raise ValueError('Select and load a trained model first.')
    X, y = data(); results = []
    for name, item in MODELS.items():
        try:
            predict(name, X.iloc[:min(len(X), 2)])
            times = []
            for _ in range(5):
                start = time.perf_counter()
                pred = predict(name, X)
                times.append((time.perf_counter()-start)*1000)
            row = dict(model=name, type=item['kind'], n=len(X),
                median_ms=round(float(np.median(times)),3))
            if task_ui.value == 'classification':
                row.update(accuracy=accuracy_score(y,pred),
                    f1=f1_score(y,pred,average='weighted',zero_division=0))
            else:
                row.update(rmse=float(np.sqrt(mean_squared_error(y,pred))),
                    r2=r2_score(y,pred))
            results.append(row)
        except Exception as e:
            results.append({'model':name,'error':str(e)})
    board = pd.DataFrame(results)
    metric = 'f1' if task_ui.value == 'classification' else 'rmse'
    if metric in board.columns:
        board = board.sort_values(metric, ascending=(metric=='rmse'), na_position='last')
    board.to_csv(OUT/'benchmark_leaderboard.csv', index=False)
    return board

app = FastAPI(title='Kaggle Multi-Model Local Test API')
class PredictRequest(BaseModel):
    instances: list[dict]

@app.get('/health')
def health(): return {'ok':True,'loaded':len(MODELS)}

@app.get('/models')
def list_models(): return [{'name':k, 'type':v['kind']} for k,v in MODELS.items()]

@app.post('/predict/{name}')
def predict_one(name: str, body: PredictRequest):
    if name not in MODELS: raise HTTPException(404,'Model not found')
    try: return {'model':name,'predictions':predict(name,pd.DataFrame(body.instances)).tolist()}
    except Exception as e: raise HTTPException(422,str(e))

@app.post('/predict-all')
def predict_all(body: PredictRequest):
    X = pd.DataFrame(body.instances)
    result = {}
    for name in MODELS:
        try: result[name] = predict(name,X).tolist()
        except Exception as e: result[name] = {'error':str(e)}
    return {'results':result}

client = TestClient(app)  # In-process HTTP test; no public Kaggle port needed.

def show_benchmark(_=None):
    with log:
        clear_output(wait=True)
        try:
            display(benchmark())
            print('Saved:', OUT/'benchmark_leaderboard.csv')
        except Exception as e: print('BENCHMARK ERROR:', e)

def query(_=None):
    with log:
        clear_output(wait=True)
        try:
            X, _ = data()
            payload = {'instances':json.loads(X.iloc[:1].to_json(orient='records'))}
            print('GET /health:', client.get('/health').json())
            print('GET /models:', client.get('/models').json())
            r = client.post('/predict-all', json=payload)
            print('POST /predict-all:',r.status_code)
            print(json.dumps(r.json(),indent=2))
        except Exception as e: print('API ERROR:',e)

refresh_btn.on_click(refresh)
load_btn.on_click(load)
check_btn.on_click(check)
bench_btn.on_click(show_benchmark)
query_btn.on_click(query)
csv_ui.observe(set_targets, names='value')
display(w.HTML('<h3>Kaggle Benchmark + FastAPI Model Lab</h3>'))
display(w.HTML('<p>Model files: .pt or .joblib only. Dataset: .csv only.</p>'))
display(models_ui,csv_ui,target_ui,task_ui)
display(w.HBox([refresh_btn,load_btn,check_btn,bench_btn,query_btn]),log)
refresh()
print('READY: Select model(s) and CSV, Load models, Check inputs, Benchmark, Query FastAPI.')
